In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import json
from itertools import product
from pathlib import Path
from itertools import product
import hashlib

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

In [ ]:
EXPERIMENT_NUMBER = 8

# -------------------------------------------------
# Region code mapping (single source of truth)
# -------------------------------------------------
REGION_CODE_MAP = {
    "CH": "11_CH",
    "NOR": "08_NOR",
    "ISL": "06_ISL",
    "CEU": "11_CEU",
}

REGION_GROUPS = {
    "CEU": ["FR", "CH", "IT_AT"],
}

SOURCE_CODES = list(REGION_CODE_MAP.keys())
TARGET_CODES = list(REGION_CODE_MAP.keys())

MODEL = "AdapterLSTM"
BASE_MODELS_DIR = Path(cfg.dataPath) / path_cache / "models"
BASE_LOGS_DIR = Path(cfg.dataPath) / path_cache / "logs"
BASE_RESULTS_DIR = Path(cfg.dataPath) / path_cache / "results"

MODELS_DIR = BASE_MODELS_DIR / f"Full_monitoring_experiment_{MODEL}_{EXPERIMENT_NUMBER}"
CACHE_DIR = BASE_LOGS_DIR / f"LSTM_cache_TL_Full_monitoring_experiment_{EXPERIMENT_NUMBER}"
RESULTS_DIR = BASE_RESULTS_DIR / f"Full_monitoring_experiment_{EXPERIMENT_NUMBER}"

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = [("ISL", "CH")]
PAIR_KEYS = ["ISL_to_CH"]

print(f"Pairs: (N = {len(PAIRS)})", PAIRS)
print("Pair keys:", PAIR_KEYS)

## Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
def resolve_region_codes(region, region_groups=None):
    if region_groups is None:
        region_groups = {}
    return region_groups.get(region, [region])


def filter_by_region(df, region, region_groups=None, source_col="SOURCE_CODE"):
    codes = resolve_region_codes(region, region_groups)
    return df[df[source_col].isin(codes)].copy()


def split_signature(glaciers):
    g = sorted(map(str, glaciers))
    return hashlib.md5((",".join(g)).encode("utf-8")).hexdigest()[:10]

## Monthly datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_Europe.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_Europe.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

MONTHLY_COLS = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
    "ELEVATION_DIFFERENCE",
]
STATIC_COLS = ["aspect", "slope", "svf"]
FEATURE_COLS = MONTHLY_COLS + STATIC_COLS

## Experiment design:

In [ ]:
# load all stake dfs once
dfs = {
    rid: load_stakes_for_rgi_region(cfg, rid)
    for rid in tqdm(RGI_REGIONS.keys(), desc="Loading stake regions")
}

# recompute only pairs involving these symbolic/original region labels
# examples:
#   None          -> load all
#   {"CEU"}       -> recompute all pairs where src=="CEU" or tgt=="CEU"
#   {"ISL"}       -> recompute all pairs involving ISL
#   {"CEU","ISL"} -> recompute all pairs involving either CEU or ISL
#   "ALL"         -> recompute all
RECOMPUTE_REGIONS = None

all_pair_res = {}
all_pair_split_info = {}


def resolve_region(code, region_groups):
    """Return either the raw code or the grouped list of codes."""
    return region_groups.get(code, code)


def should_recompute_pair(src, tgt, recompute_regions=None):
    """
    Decide whether a pair should be recomputed.

    Parameters
    ----------
    src, tgt : str
        Pair labels as they appear in PAIRS, e.g. "CH", "ISL", "CEU".
    recompute_regions : set[str] | None | str
        - None  -> load all pairs
        - "ALL" -> recompute all pairs
        - set   -> recompute only pairs where src or tgt is in this set
    """
    if recompute_regions is None:
        return False
    if recompute_regions == "ALL":
        return True
    return (src in recompute_regions) or (tgt in recompute_regions)


for src, tgt in tqdm(PAIRS, desc="Preparing pairwise monthlies"):
    key = f"{src}_to_{tgt}"

    src_codes = resolve_region(src, REGION_GROUPS)
    tgt_codes = resolve_region(tgt, REGION_GROUPS)

    recompute_this = should_recompute_pair(
        src=src,
        tgt=tgt,
        recompute_regions=RECOMPUTE_REGIONS,
    )

    res_xreg, split_info = prepare_monthly_df_xreg_pairwise(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        source_code=src_codes,
        target_code=tgt_codes,
        run_flag=recompute_this,  # True = recompute, False = load
        region_name=f"XREG_{src}_TO_{tgt}",
        csv_subfolder=f"CrossRegional/{src}_to_{tgt}/csv",
    )

    all_pair_res[key] = res_xreg
    all_pair_split_info[key] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    status = "recomputed" if recompute_this else "loaded"

    print("\n" + "=" * 80)
    print(f"Pair: {key} [{status}]")
    print(f"Train glaciers ({src}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers ({tgt}):", len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "| Test rows:", len(df_test))

In [ ]:
df_sc_check = summarize_source_code_issues_in_all_pair_res(all_pair_res)
df_sc_check

In [ ]:
for key, res in all_pair_res.items():
    src_label, tgt_label = key.split("_to_")

    src_codes = resolve_region_codes(src_label, REGION_GROUPS)
    tgt_codes = resolve_region_codes(tgt_label, REGION_GROUPS)

    df_test = res["df_test"]
    df_train = res["df_train"]

    # In pairwise setup, df_test should already be target-only and df_train source-only,
    # but we still filter explicitly for robustness.
    df_tgt = df_test[df_test["SOURCE_CODE"].isin(tgt_codes)].copy()
    df_src = df_train[df_train["SOURCE_CODE"].isin(src_codes)].copy()

    print("\n" + "=" * 80)
    print(f"Pair: {src_label} → {tgt_label}")

    print(f"{src_label} monthly rows:", len(df_src))
    print(f"{src_label} glaciers:", df_src["GLACIER"].nunique())
    if len(df_src) > 0:
        print(
            f"{src_label} year range:",
            df_src["YEAR"].min(),
            "-",
            df_src["YEAR"].max(),
        )
    else:
        print(f"No source rows found for {src_label}.")

    print(f"{tgt_label} monthly rows:", len(df_tgt))
    print(f"{tgt_label} glaciers:", df_tgt["GLACIER"].nunique())
    if len(df_tgt) > 0:
        print(
            f"{tgt_label} year range:",
            df_tgt["YEAR"].min(),
            "-",
            df_tgt["YEAR"].max(),
        )
    else:
        print(f"No target rows found for {tgt_label}.")

### Fixed glacier hold-out split (spatial generalization):

In [ ]:
def mmd_squared_unbiased(
    X: np.ndarray,
    Y: np.ndarray,
    bandwidths: list[float] | None = None,
    max_samples: int = 5000,
    seed: int = 0,
) -> float:
    """
    Unbiased estimator of MMD^2 using a mixture of RBF kernels (MK-MMD).

    MMD^2(P,Q) = E[k(x,x')] + E[k(y,y')] - 2*E[k(x,y)]

    If bandwidths is None, uses the median heuristic on the pooled sample
    and adds two extra scales (0.5x and 2x) for robustness.
    """
    if X.size == 0 or Y.size == 0:
        return np.nan

    rng = np.random.default_rng(seed)

    def _subsample(A: np.ndarray) -> np.ndarray:
        if A.shape[0] <= max_samples:
            return A
        return A[rng.choice(A.shape[0], max_samples, replace=False)]

    X, Y = _subsample(X), _subsample(Y)
    m, n = X.shape[0], Y.shape[0]

    if bandwidths is None:
        # Median heuristic on pooled sample
        Z = np.vstack([X, Y])
        if Z.shape[0] > 2000:
            Z = Z[rng.choice(Z.shape[0], 2000, replace=False)]
        z2 = np.sum(Z * Z, axis=1, keepdims=True)
        D2 = np.maximum(z2 + z2.T - 2.0 * (Z @ Z.T), 0.0)
        np.fill_diagonal(D2, 0.0)
        median_d2 = np.median(D2[D2 > 0])
        sig = float(np.sqrt(0.5 * median_d2))
        bandwidths = [sig * 0.5, sig, sig * 2.0]

    def _rbf(A: np.ndarray, B: np.ndarray, sigma: float) -> np.ndarray:
        a2 = np.sum(A * A, axis=1, keepdims=True)
        b2 = np.sum(B * B, axis=1, keepdims=True).T
        return np.exp(-np.maximum(a2 + b2 - 2.0 * (A @ B.T), 0.0) /
                      (2.0 * sigma**2))

    mmd2_per_kernel = []
    for sigma in bandwidths:
        Kxx = _rbf(X, X, sigma)
        np.fill_diagonal(Kxx, 0.0)
        Kyy = _rbf(Y, Y, sigma)
        np.fill_diagonal(Kyy, 0.0)
        Kxy = _rbf(X, Y, sigma)
        mmd2_per_kernel.append(Kxx.sum() / (m * (m - 1)) + Kyy.sum() /
                               (n * (n - 1)) - 2.0 * Kxy.mean())

    return float(np.mean(mmd2_per_kernel))

In [ ]:
def compute_domain_shift(
    df_src: pd.DataFrame,
    df_tgt: pd.DataFrame,
    monthly_cols: list[str],
    static_cols: list[str],
    id_col: str = "ID",
    seed: int = 0,
) -> dict:
    """
    Assess input-space domain shift between source (Iceland) and target (CH).
    
    Aggregates to measurement level (one row per ID) before computing
    distances, avoiding pseudoreplication from repeated static features
    and variable-length month sequences.
    """

    def _pivot(df):
        g = df.groupby(id_col)
        Xm = g[monthly_cols].mean().to_numpy(dtype=np.float64)
        Xs = g[static_cols].first().to_numpy(dtype=np.float64)
        return Xm, Xs

    Xm_src, Xs_src = _pivot(df_src)
    Xm_tgt, Xs_tgt = _pivot(df_tgt)

    # Fit scaler on source+target combined — same space for both
    from sklearn.preprocessing import StandardScaler

    scaler_m = StandardScaler().fit(np.vstack([Xm_src, Xm_tgt]))
    scaler_s = StandardScaler().fit(np.vstack([Xs_src, Xs_tgt]))

    Xm_src_z = scaler_m.transform(Xm_src)
    Xm_tgt_z = scaler_m.transform(Xm_tgt)
    Xs_src_z = scaler_s.transform(Xs_src)
    Xs_tgt_z = scaler_s.transform(Xs_tgt)

    # Joint climate + topo
    X_src_z = np.hstack([Xm_src_z, Xs_src_z])
    X_tgt_z = np.hstack([Xm_tgt_z, Xs_tgt_z])

    out = {
        "n_src": len(Xm_src),
        "n_tgt": len(Xm_tgt),
        # --- joint ---
        "D_mmd2_joint": mmd_squared_unbiased(X_src_z, X_tgt_z, seed=seed),
        "D_energy_joint": energy_distance(X_src_z, X_tgt_z, seed=seed + 1),
        # --- climate only ---
        "D_mmd2_climate": mmd_squared_unbiased(Xm_src_z,
                                               Xm_tgt_z,
                                               seed=seed + 2),
        "D_energy_climate": energy_distance(Xm_src_z, Xm_tgt_z, seed=seed + 3),
        # --- topo only ---
        "D_mmd2_topo": mmd_squared_unbiased(Xs_src_z, Xs_tgt_z, seed=seed + 4),
        "D_energy_topo": energy_distance(Xs_src_z, Xs_tgt_z, seed=seed + 5),
    }

    # --- per-variable marginal MMD (most interpretable diagnostic) ---
    all_cols = monthly_cols + static_cols
    X_src_all = np.hstack([Xm_src_z, Xs_src_z])
    X_tgt_all = np.hstack([Xm_tgt_z, Xs_tgt_z])
    for j, col in enumerate(all_cols):
        out[f"D_mmd2_{col}"] = float(
            mmd_squared_unbiased(
                X_src_all[:, j:j + 1],
                X_tgt_all[:, j:j + 1],
                seed=seed + 10 + j,
            ))

    return out

In [ ]:
shift = compute_domain_shift(
    df_src=res["df_train"],  # Iceland
    df_tgt=res["df_test"],  # Switzerland
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

In [ ]:
# Summary distances
for k in [
        "D_mmd2_joint", "D_mmd2_climate", "D_mmd2_topo", "D_energy_joint",
        "D_energy_climate", "D_energy_topo"
]:
    print(f"{k:30s}: {shift[k]:.4f}")

# Per-variable ranking (most shifted first)
var_shifts = {
    k: v
    for k, v in shift.items() if k.startswith("D_mmd2_")
    and k not in ["D_mmd2_joint", "D_mmd2_climate", "D_mmd2_topo"]
}
for k, v in sorted(var_shifts.items(), key=lambda x: -x[1]):
    print(f"  {k:30s}: {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_domain_shift(shift: dict, monthly_cols: list[str], static_cols: list[str]):
    
    # --- data ---
    var_keys = {
        f"D_mmd2_{col}": col for col in monthly_cols + static_cols
    }
    records = [
        (label, shift[key])
        for key, label in var_keys.items()
        if key in shift
    ]
    records.sort(key=lambda x: x[1], reverse=True)
    labels, values = zip(*records)

    # --- colors by physical group ---
    radiation_vars = {"sshf", "slhf", "ssrd", "str", "fal"}
    topo_vars      = {"slope", "svf", "aspect", "ELEVATION_DIFFERENCE"}

    def _color(label):
        if label in radiation_vars: return "#D85A30"
        if label in topo_vars:      return "#534AB7"
        return "#888780"

    colors = [_color(l) for l in labels]

    # --- plot ---
    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 5),
        gridspec_kw={"width_ratios": [1, 2.2]},
    )

    # ── Left: summary bars (joint / climate / topo) ──────────────────────────
    ax_left = axes[0]
    summary_labels = ["joint", "climate", "topo"]
    summary_vals   = [
        shift["D_mmd2_joint"],
        shift["D_mmd2_climate"],
        shift["D_mmd2_topo"],
    ]
    summary_colors = ["#3d3d3a", "#D85A30", "#534AB7"]

    bars = ax_left.barh(
        summary_labels, summary_vals,
        color=summary_colors, height=0.5,
    )
    ax_left.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax_left.set_xlim(0, max(summary_vals) * 1.35)
    ax_left.set_xlabel("MMD²", fontsize=11)
    ax_left.set_title("Summary", fontsize=12, fontweight="normal", pad=10)
    ax_left.invert_yaxis()
    ax_left.spines[["top", "right"]].set_visible(False)
    ax_left.grid(axis="x", color="#e0e0e0", linewidth=0.6)
    ax_left.set_axisbelow(True)

    # ── Right: per-variable bars ──────────────────────────────────────────────
    ax_right = axes[1]
    y = np.arange(len(labels))

    bars2 = ax_right.barh(y, values, color=colors, height=0.6)
    ax_right.bar_label(bars2, fmt="%.3f", padding=4, fontsize=9)
    ax_right.set_yticks(y)
    ax_right.set_yticklabels(labels, fontsize=10)
    ax_right.set_xlim(0, max(values) * 1.25)
    ax_right.set_xlabel("MMD²", fontsize=11)
    ax_right.set_title("Per-variable shift  (Iceland → Switzerland)", fontsize=12, fontweight="normal", pad=10)
    ax_right.invert_yaxis()
    ax_right.spines[["top", "right"]].set_visible(False)
    ax_right.grid(axis="x", color="#e0e0e0", linewidth=0.6)
    ax_right.set_axisbelow(True)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(color="#D85A30", label="radiation / energy flux"),
        mpatches.Patch(color="#534AB7", label="topographic"),
        mpatches.Patch(color="#888780", label="temperature / precip"),
    ]
    ax_right.legend(
        handles=legend_handles,
        loc="lower right",
        frameon=False,
        fontsize=9,
    )

    fig.tight_layout(w_pad=3)
    return fig

In [ ]:
fig = plot_domain_shift(shift, MONTHLY_COLS, STATIC_COLS)
fig.savefig("domain_shift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
gfeat = glacier_level_features(df_isl)
print("Number of glaciers:", len(gfeat))
gfeat.head()

In [ ]:
df_isl_pool, df_isl_holdout, holdout_glaciers, pool_glaciers, gfeat, split_summary = (
    holdout_split_cluster_stratified(df_isl,
                                     holdout_frac=0.30,
                                     seed=cfg.seed,
                                     n_clusters=6))

print(split_summary)

In [ ]:
g_hold = gfeat[gfeat["GLACIER"].isin(holdout_glaciers)]
g_pool = gfeat[gfeat["GLACIER"].isin(pool_glaciers)]

for col in ["slope_mean", "slope_std", "svf_mean", "nyears", "nrows"]:
    print(col)
    print("  pool   :", g_pool[col].quantile([0.1, 0.5, 0.9]).to_dict())
    print("  holdout:", g_hold[col].quantile([0.1, 0.5, 0.9]).to_dict())

In [ ]:
print("Cluster counts - pool")
print(g_pool["cluster"].value_counts().sort_index())

print("\nCluster counts - holdout")
print(g_hold["cluster"].value_counts().sort_index())

#### Plot location:

In [ ]:
ft_glaciers_by_split = {
    "spatial": holdout_glaciers,
}

data_ISL, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=["spatial"],
    ft_label_col="Pool/Hold-out glacier",
    ft_label_ft="Pool",
    ft_label_holdout="Hold-out",
)

glacier_df_ISL_5pct = glacier_info_by_split["spatial"]

cmap_for_train = cm.batlow
train_color = "#1f4e79"
# requires your helper
colors = get_cmap_hex(cmap_for_train, 10)  # noqa: F821
train_color = colors[0]

palette = {"Hold-out": train_color, "Pool": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL_5pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Iceland",
    extent=(-25, -11, 62, 68),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,  # optional, uses your get_cmap_hex if available
    split_col="Pool/Hold-out glacier")

#### Plot distributions overlap:

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

In [ ]:
feature_columns_ = feature_columns + ['POINT_BALANCE']
fig = plot_feature_kde_overlap(df_isl_pool,
                               df_isl_holdout,
                               feature_columns_,
                               label_dist_1="Pool",
                               label_dist_2="Hold-out")

#### Plot overlap CH & Iceland:

In [ ]:
# res_xreg is the ONE dict from your cross-regional monthly prep
figs_by_code = plot_tsne_overlap_xreg_from_single_res(
    res_xreg=res_xreg,
    cfg=cfg,
    STATIC_COLS=STATIC_COLS,
    MONTHLY_COLS=MONTHLY_COLS,
    group_col="SOURCE_CODE",
    target_code="CH",
    use_aug=False,  # or True if you want *_aug
    n_iter=1000,
    only_codes=["ISL"],  # optional
)

### Build static assets once (CH + fixed ISL holdout)

In [ ]:
import hashlib


def split_signature(glaciers):
    g = sorted(map(str, glaciers))
    return hashlib.md5((",".join(g)).encode("utf-8")).hexdigest()[:10]


print("CH_to_ISL", "holdout_sig:", split_signature(holdout_glaciers),
      "n_hold:", len(holdout_glaciers))

In [ ]:
sig = split_signature(holdout_glaciers)
cache_dir_pair = CACHE_DIR / f"holdout_{sig}"
cache_dir_pair.mkdir(parents=True, exist_ok=True)

static_assets = build_static_tl_assets_source_and_holdout(
    cfg=cfg,
    res_xreg=res_xreg,
    target_code="ISL",
    source_code="CH",
    holdout_glaciers=holdout_glaciers,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=cache_dir_pair,
    force_recompute=False,
    show_progress=True)

### Monitoring subsamples:

#### Define the pools for experiments:

In [ ]:
df_isl_pool_rows = df_isl[df_isl["GLACIER"].isin(pool_glaciers)]

df_pool_monthly = df_isl_pool_rows.copy()

G_max = df_pool_monthly["GLACIER"].nunique()
Y_max = df_pool_monthly.groupby(
    "GLACIER")["YEAR"].nunique().max()  # max years available on any glacier

# How many rows per glacier-year exist? (this sets an upper bound for M)
df_isl_pool_rows_raw = dfs["06"]
df_isl_pool_rows_raw = df_isl_pool_rows_raw[
    df_isl_pool_rows_raw["GLACIER"].isin(pool_glaciers)]

rows_per_gy = df_isl_pool_rows_raw.groupby(["GLACIER", "YEAR"]).size()
M_p25 = int(rows_per_gy.quantile(0.25))
M_p50 = int(rows_per_gy.median())
M_p90 = int(rows_per_gy.quantile(0.90))
M_max = int(rows_per_gy.max())
M_min = int(rows_per_gy.min())

print("POOL CAPACITY")
print("G_max:", G_max)
print("Y_max (max years on a glacier):", Y_max)
print("Rows per glacier-year (monthly format): min", M_min, "| p25", M_p25,
      "| median", M_p50, "| p90", M_p90, "| max", M_max)

In [ ]:
# -----------------------
# Budgets & seeds (global)
# -----------------------
G_set = [1, 2, 3, 5, 10, 15, 19]  # (19 is max accross all regions)
Y_set = [1, 2, 3, 5, 10, 20, 30, 35]
BUDGETS = [{"G": int(g), "Y": int(y)} for g in G_set for y in Y_set]

N_SEEDS = 10
SEEDS = list(range(1, N_SEEDS + 1))

print("Total budget points:", len(BUDGETS))
print(pd.DataFrame(BUDGETS).sort_values(["G", "Y"]).head(12))
print("Seeds:", SEEDS)


In [ ]:
# Sanity check: pick one experiment and look at chosen glaciers and years
df_ft, chosen, picked_years = sample_monitoring_subset_from_pool(
    df_pool=df_isl_pool_rows,
    G=5,
    Y=5,
    seed=cfg.seed,
    glacier_pick_method="random",
    year_pick_method="earliest_block",
    enforce_full_Y_if_possible=True,
)
print("Chosen glaciers:", sorted(chosen))
print("\nYears per glacier:")

for gid in sorted(chosen):
    print(gid, picked_years.get(gid))

In [ ]:
# Sanity check: pick one experiment and look at chosen glaciers and years
chosen_by_seed = {}
for seed in [0, 1, 2, 3, 4]:
    _, chosen, _ = sample_monitoring_subset_from_pool(
        df_pool=df_isl_pool_rows,
        G=5,
        Y=5,
        seed=seed,
        glacier_pick_method="random",
        year_pick_method="earliest_block",
    )
    chosen_by_seed[seed] = set(chosen)

for s, ch in chosen_by_seed.items():
    print(f"Seed {s}: {sorted(ch)}")

In [ ]:
# Build list of tasks first
TASKS = [(b, seed) for b in BUDGETS for seed in SEEDS]

print("Total experiments to build:", len(TASKS))

assets_all = {}

for b, seed in tqdm(TASKS, desc="Building LSTM assets"):

    df_ft, chosen, _ = sample_monitoring_subset_from_pool(
        df_pool=df_isl_pool_rows,
        G=b["G"],
        Y=b["Y"],
        seed=seed,
        glacier_pick_method="random",
        year_pick_method="earliest_block",
        enforce_full_Y_if_possible=True,
    )

    exp_key = f"TL_CH_to_ISL_G{b['G']}_Y{b['Y']}_seed{seed}"
    sig = split_signature(holdout_glaciers)
    cache_dir_pair = CACHE_DIR / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)
    assets_one = build_budget_assets_finetune_only(cfg=cfg,
                                                   res_xreg=res_xreg,
                                                   static_assets=static_assets,
                                                   df_ft=df_ft,
                                                   exp_key=exp_key,
                                                   MONTHLY_COLS=MONTHLY_COLS,
                                                   STATIC_COLS=STATIC_COLS,
                                                   cache_dir=cache_dir_pair,
                                                   force_recompute=False,
                                                   val_ratio=0.2,
                                                   show_progress=False)

    assets_all.update(assets_one)

print("Total experiments built:", len(assets_all))

### LSTM CH Baseline:

In [ ]:
log_path_gs_results = {
    "ISL": 'logs/GS_results/lstm_param_search_progress_OOS_ISL_2026-02-11.csv',
    "NOR": 'logs/GS_results/lstm_param_search_progress_OOS_NOR_2026-02-09.csv',
    "FR": 'logs/GS_results/lstm_param_search_progress_OOS_FR_2026-02-06.csv',
    "CH": 'logs/GS_results/lstm_param_search_progress_CH_2026-02-18.csv',
}

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)


In [ ]:
tl_assets_static = {"STATIC": static_assets}
model_ch, ch_path, ch_info = train_or_load_source_baseline(
    cfg=cfg,
    source_code="CH",
    tl_assets=tl_assets_static,
    default_params=params_by_key["11_CH"],
    device=device,
    models_dir=MODELS_DIR,
    prefix="lstm_CH",
    key="BASELINE",
    train_flag=True,  # or False to only load
    force_retrain=False,
    epochs=150,
    batch_size_train=64,
    batch_size_val=128,
    verbose=False,
    date=None)

### Compute E_ZERO:
E_ZERO = error of the CH baseline model evaluated on the fixed ISL holdout dataset (unseen glaciers), with no finetuning

In [ ]:
# Make a tl_assets dict with one key (because your codebase often expects dict-of-keys)
tl_assets_zero = {"TL_CH_to_ISL_ZERO": static_assets}

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

metrics_zero, df_preds_zero, _, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=model_ch,  # <-- CH baseline model
    device=device,
    tl_assets_for_key=tl_assets_zero["TL_CH_to_ISL_ZERO"],
    ax=ax,
    title="E_ZERO: CH baseline on ISL holdout",
    batch_size=128,
    domain_vocab=tl_assets_zero["TL_CH_to_ISL_ZERO"].get("domain_vocab", None),
)

plt.show()

E_ZERO = np.mean([metrics_zero["RMSE_annual"], metrics_zero["RMSE_winter"]])
E_ZERO_a = metrics_zero["RMSE_annual"]
E_ZERO_w = metrics_zero["RMSE_winter"]
print("E_ZERO_a (RMSE_annual):", E_ZERO_a)
print("E_ZERO_w (RMSE_winter):", E_ZERO_w)
print("E_ZERO (mean of annual and winter RMSE):", E_ZERO)
print(metrics_zero)

### E_TL:

In [ ]:
# load GS results:
with open("results/tuning_adapter/adapter_best_by_region_2026-02-24_15.json",
          "r") as f:
    best_by_region = json.load(f)

best_by_region

#### Train adapter-only models for experiments:

In [ ]:
sig = split_signature(holdout_glaciers)
model_dir_pair = MODELS_DIR / f"holdout_{sig}"
model_dir_pair.mkdir(parents=True, exist_ok=True)

models_tl_adapter, infos_tl_adapter = finetune_TL_models_all(
    cfg=cfg,
    tl_assets_by_key=assets_all,
    best_params=params_by_key["11_CH"],
    device=device,
    pretrained_ckpt_path=ch_path,
    strategies=("adapter", ),
    force_retrain=False,
    models_dir=model_dir_pair,
    prefix="lstm_TL",
    verbose=False,
    best_by_region=best_by_region,
    date=None,  # optional: to load old dates
    load_latest=True,
    skip_if_missing=False,
    prefer_tuned_ckpt=False)

#### Evaluate E_TL:
For each budget point on the same holdout

In [ ]:
RUN_EVAL = False
if RUN_EVAL:
    rows = []
    for exp_key in tqdm(sorted(assets_all.keys()),
                        desc="Evaluating TL models"):
        run_key = f"{exp_key}__adapter"
        model = models_tl_adapter.get(run_key, None)
        if model is None:
            print(f"Model for {run_key} not found, skipping evaluation."
                  )  # <-- better logging
            # checkpoint might not exist / training skipped
            continue

        assets = assets_all[exp_key]

        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=model,
            device=device,
            tl_assets_for_key=assets,
            ax=None,
            title=None,
            batch_size=128,
            domain_vocab=assets.get("domain_vocab", None),
            show_plot=False)

        metrics["exp_key"] = exp_key
        rows.append(metrics)

    df_etl_adapter = pd.DataFrame(rows).set_index("exp_key").sort_index()
    df_etl_adapter["E_ZERO_RMSE_annual"] = E_ZERO_a
    df_etl_adapter[
        "Delta_vs_ZERO_annual"] = df_etl_adapter["RMSE_annual"] - E_ZERO_a

    df_etl_adapter["E_ZERO_RMSE_winter"] = E_ZERO_w
    df_etl_adapter[
        "Delta_vs_ZERO_winter"] = df_etl_adapter["RMSE_winter"] - E_ZERO_w

    # save to csv
    df_etl_adapter.to_csv(os.path.join(RESULTS_DIR, "df_etl_adapter.csv"))
else:
    df_etl_adapter = pd.read_csv(os.path.join(RESULTS_DIR,
                                              "df_etl_adapter.csv"),
                                 index_col="exp_key")

display(df_etl_adapter)

In [ ]:
# exp_key now has no _M... part
_re_budget = re.compile(r"_G(?P<G>\d+)_Y(?P<Y>\d+)_seed(?P<seed>\d+)")


def parse_budget(s: str):
    m = _re_budget.search(str(s))
    if not m:
        return {"G": np.nan, "Y": np.nan, "seed": np.nan}
    return {k: int(v) for k, v in m.groupdict().items()}


# keep exp_key as a column (don’t rely on rename quirks)
df_etl_adapter2 = df_etl_adapter.reset_index()  # creates column "exp_key"
meta = df_etl_adapter2["exp_key"].apply(parse_budget).apply(pd.Series)
df_etl_adapter2 = pd.concat([df_etl_adapter2, meta], axis=1)

# drop non-budget rows
df_etl_adapter2 = df_etl_adapter2.dropna(subset=["G", "Y", "seed"]).copy()

# aggregate across seeds per (G,Y)
agg = df_etl_adapter2.groupby(["G", "Y"]).agg(
    # ---- Annual ----
    RMSE_a_med=("RMSE_annual", "median"),
    RMSE_a_p10=("RMSE_annual", lambda x: np.quantile(x, 0.10)),
    RMSE_a_p90=("RMSE_annual", lambda x: np.quantile(x, 0.90)),

    # ---- Winter ----
    RMSE_w_med=("RMSE_winter", "median"),
    RMSE_w_p10=("RMSE_winter", lambda x: np.quantile(x, 0.10)),
    RMSE_w_p90=("RMSE_winter", lambda x: np.quantile(x, 0.90)),
    n=("RMSE_annual", "size"),
).reset_index().sort_values(["G", "Y"])

display(agg)

### Compute E_FULL: 
Adapter fine-tuned on all ISL pool data (everything that is not in the fixed holdout glaciers), evaluated on the same fixed holdout (ds_test).

In [ ]:
exp_key_full = "TL_CH_to_ISL_FULLPOOL"

assets_full = build_budget_assets_finetune_only(
    cfg=cfg,
    res_xreg=res_xreg,
    static_assets=static_assets,
    df_ft=df_isl_pool_rows,  # <-- ALL pool data
    exp_key=exp_key_full,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=CACHE_DIR,
    force_recompute=False,
    val_ratio=0.2,
)

# merge into your experiment dict (optional but convenient)
assets_all_plus = dict(assets_all)
assets_all_plus.update(assets_full)

print("FULL asset built:", exp_key_full)
print("Full finetune sequences:",
      len(assets_all_plus[exp_key_full]["ds_finetune"]))

In [ ]:
models_full, infos_full = finetune_TL_models_all(
    cfg=cfg,
    tl_assets_by_key={exp_key_full:
                      assets_all_plus[exp_key_full]},  # only this one
    best_params=params_by_key["11_CH"],
    device=device,
    pretrained_ckpt_path=ch_path,
    strategies=("adapter", ),
    force_retrain=False,
    models_dir=MODELS_DIR,
    prefix="lstm_TL",
    verbose=False,
    best_by_region=None,
    date="fixed",
)

run_key_full = f"{exp_key_full}__adapter"
model_full = models_full[run_key_full]  # <-- this is the model object

metrics_full, df_preds_full, _, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=model_full,
    device=device,
    tl_assets_for_key=assets_all_plus[exp_key_full],
    ax=None,
    title=None,
    batch_size=128,
    domain_vocab=assets_all_plus[exp_key_full].get("domain_vocab", None),
)

E_FULL_a = metrics_full["RMSE_annual"]
E_FULL_w = metrics_full["RMSE_winter"]
E_FULL = np.mean([E_FULL_a, E_FULL_w])

print("E_FULL (mean annual and winter):", E_FULL)
print("E_FULL_a (RMSE_annual):", E_FULL_a)
print("E_FULL_w (RMSE_winter):", E_FULL_w)
print(metrics_full)

### Compute E_SCRATCH:
“no-transfer” (from-scratch) baseline trained on the same small ISL monitoring subset and evaluated on the same fixed ISL holdout

In [ ]:
RUN_SCRATCH = False

if RUN_SCRATCH:
    _re_budget = re.compile(
        r"_G(?P<G>\d+)_Y(?P<Y>\d+)_M(?P<M>\d+)_seed(?P<seed>\d+)")

    def parse_budget(exp_key: str):
        m = _re_budget.search(str(exp_key))  # works even with ..._FT_ft.joblib
        if not m:
            return {"G": np.nan, "Y": np.nan, "M": np.nan, "seed": np.nan}
        return {k: int(v) for k, v in m.groupdict().items()}

    def within_assets_from_tl_assets(tl_assets_for_key: dict):
        return {
            "ds_train": tl_assets_for_key["ds_finetune"],
            "ds_test": tl_assets_for_key["ds_test"],
            "train_idx": tl_assets_for_key["finetune_train_idx"],
            "val_idx": tl_assets_for_key["finetune_val_idx"],
        }

    models_within = {}
    infos_within = {}
    rows = []

    exp_keys = sorted(assets_all.keys())
    pbar = tqdm(exp_keys,
                desc="Training+Evaluating E_SCRATCH",
                dynamic_ncols=True)

    for exp_key in pbar:
        meta = parse_budget(exp_key)
        if not np.isnan(meta["G"]):
            pbar.set_postfix(meta)

        w_assets = within_assets_from_tl_assets(assets_all[exp_key])

        model_w, path_w, info_w = train_or_load_one_within_region(
            cfg=cfg,
            key=exp_key,
            lstm_assets=w_assets,
            best_params=params_by_key["06_ISL"],
            device=device,
            models_dir=MODELS_DIR,
            prefix="lstm_within_ISL",
            train_flag=False,
            force_retrain=False,
            epochs=150,
            batch_size_train=64,
            batch_size_val=128,
            batch_size_test=128,
            date="2026-02-27",
            verbose=False)

        models_within[exp_key] = model_w
        infos_within[exp_key] = {"model_path": path_w, **(info_w or {})}

        # ---- Evaluate ----
        met_w, df_w = model_w.evaluate_with_preds(
            device,
            info_w["test_dl"],
            info_w["ds_test"],
        )

        # ---- Seasonal metrics (annual + winter) ----
        scores_annual, scores_winter = compute_seasonal_scores(
            df_w, target_col="target", pred_col="pred")

        rows.append({
            "exp_key":
            exp_key,

            # scratch annual/winter
            "RMSE_SCRATCH_annual":
            float(scores_annual["rmse"]),
            "RMSE_SCRATCH_winter":
            float(scores_winter["rmse"]),
            "R2_SCRATCH_annual":
            float(scores_annual["R2"]),
            "R2_SCRATCH_winter":
            float(scores_winter["R2"]),
            "Bias_SCRATCH_annual":
            float(scores_annual["Bias"]),
            "Bias_SCRATCH_winter":
            float(scores_winter["Bias"]),
            "n_annual":
            int(scores_annual.get("n", np.nan)) if isinstance(
                scores_annual, dict) else np.nan,
            "n_winter":
            int(scores_winter.get("n", np.nan)) if isinstance(
                scores_winter, dict) else np.nan,
            **meta,
            "model_path":
            path_w,
        })

    df_scratch = pd.DataFrame(rows).set_index("exp_key").sort_index()

    # save
    df_scratch.to_csv(
        f"results/ISL_CH_experiment_{EXPERIMENT_NUMBER}/E_SCRATCH.csv")
    display(df_scratch.head())
else:
    df_scratch = None

## Evaluate experiments:

### Build one unified results dataframe (with G/Y/M/seed parsed):

In [ ]:
# 1) Parse G/Y/seed from *any* exp_key string (even if it has suffixes)
_re_budget = re.compile(r"_G(?P<G>\d+)_Y(?P<Y>\d+)_seed(?P<seed>\d+)")


def parse_budget(s: str):
    m = _re_budget.search(str(s))
    if not m:
        return {"G": np.nan, "Y": np.nan, "seed": np.nan}
    return {k: int(v) for k, v in m.groupdict().items()}


# 2) Start from TL results (has RMSE_annual/RMSE_winter etc.)
df = df_etl_adapter.copy()

# 3) Optionally join scratch results (if provided)
scratch_cols = [
    "RMSE_SCRATCH_annual",
    "RMSE_SCRATCH_winter",
    "R2_SCRATCH_annual",
    "R2_SCRATCH_winter",
    "Bias_SCRATCH_annual",
    "Bias_SCRATCH_winter",
]

df_scratch = None
if "df_scratch" in globals() and df_scratch is not None:
    scratch_cols_present = [c for c in scratch_cols if c in df_scratch.columns]
    if len(scratch_cols_present) > 0:
        # prefer inner join to keep only exp_keys available in both
        df = df.join(df_scratch[scratch_cols_present], how="inner")
    else:
        print(
            "df_scratch provided but none of the expected scratch columns are present; skipping join."
        )
else:
    df_scratch = None  # ensure name exists
    print("df_scratch not provided; proceeding without scratch metrics.")

# 4) Add parsed budget columns
meta = df.index.to_series().apply(parse_budget).apply(pd.Series)
df = pd.concat([meta, df], axis=1)

# 5) Drop any rows that didn’t parse (e.g. FULLPOOL keys)
df = df.dropna(subset=["G", "Y", "seed"]).copy()

# 6) Derived quantities for plotting (no M anymore)
df["Effort"] = df["G"] * df["Y"]

# Baselines (already computed earlier in notebook)
df["E_ZERO_a"] = float(E_ZERO_a)
df["E_ZERO_w"] = float(E_ZERO_w)
df["E_ZERO"] = float(E_ZERO)  # mean(annual,winter)

df["E_FULL_a"] = float(E_FULL_a)
df["E_FULL_w"] = float(E_FULL_w)
df["E_FULL"] = float(E_FULL)  # mean(annual,winter)

# --- TL combined metrics ---
df["RMSE_TL"] = 0.5 * (df["RMSE_annual"] + df["RMSE_winter"])

# --- Scratch-derived metrics only if scratch columns exist ---
has_scratch = (("RMSE_SCRATCH_annual" in df.columns)
               and ("RMSE_SCRATCH_winter" in df.columns))

if has_scratch:
    df["RMSE_SCRATCH"] = 0.5 * (df["RMSE_SCRATCH_annual"] +
                                df["RMSE_SCRATCH_winter"])

    # Transfer gain (positive = TL better)
    df["Transfer_gain_annual"] = df["RMSE_SCRATCH_annual"] - df["RMSE_annual"]
    df["Transfer_gain_winter"] = df["RMSE_SCRATCH_winter"] - df["RMSE_winter"]
    df["Transfer_gain"] = df["RMSE_SCRATCH"] - df["RMSE_TL"]
else:
    print(
        "Scratch RMSE columns missing; skipping RMSE_SCRATCH and Transfer_gain columns."
    )

# Deltas vs ZERO (annual/winter + combined)
df["Delta_vs_ZERO_annual"] = df["RMSE_annual"] - df["E_ZERO_a"]
df["Delta_vs_ZERO_winter"] = df["RMSE_winter"] - df["E_ZERO_w"]
df["Delta_vs_ZERO"] = df["RMSE_TL"] - df["E_ZERO"]

# Recovery (0 = ZERO, 1 = FULL)
den = (df["E_ZERO"] - df["E_FULL"])
df["Recovery_TL"] = (df["E_ZERO"] - df["RMSE_TL"]) / den
df["Recovery_TL"] = df["Recovery_TL"].clip(-0.25, 1.25)

df.head()

### Tier sweep plots (which knob matters most?)

In [ ]:
# Run
plot_sweep_Y_by_G_colormap(
    df,
    E_ZERO_a,
    E_FULL_a,
    E_ZERO_w,
    E_FULL_w,
    E_ZERO,
    E_FULL,
    metric="mean",
    title=
    f"Iceland monitoring perf. for mean seasonal PMB, source: CH, model: adapter (K={N_SEEDS})"
)

In [ ]:
# Run
plot_sweep_Y_by_G_colormap(
    df,
    E_ZERO_a,
    E_FULL_a,
    E_ZERO_w,
    E_FULL_w,
    E_ZERO,
    E_FULL,
    metric="annual",
    title=
    f"Iceland monitoring perf. for annual PMB, source: CH, model: adapter (K={N_SEEDS})"
)

In [ ]:
# Run
plot_sweep_Y_by_G_colormap(
    df,
    E_ZERO_a,
    E_FULL_a,
    E_ZERO_w,
    E_FULL_w,
    E_ZERO,
    E_FULL,
    metric="winter",
    title=
    f"Iceland monitoring perf. for winter PMB, source: CH, model: adapter (K={N_SEEDS})"
)